# C. Chuẩn hóa OSM và dựng mạng xe máy

Ghép các ô tải, khử trùng `(type, id)`, dựng đoạn đường theo node OSM, tạo cung có hướng và quy tắc rẽ. Dữ liệu gốc trong `work/data/hanoi_tiles` được giữ nguyên.

Mạng tự động là **mạng OSM tạm thời**. Quyền đi có điều kiện, barrier thiếu chứng cứ và hạn chế rẽ chưa xử lý được có trạng thái riêng. Có thể dùng Python thường hoặc kernel PyQGIS đã đăng ký.


In [ ]:
from pathlib import Path
import sys
import json
import pandas as pd

candidates = [Path.cwd(), *Path.cwd().parents]
candidates += [p / "work" for p in candidates]
candidates += [Path.home() / "python" / "work"]
WORK = next((p for p in candidates if (p / "scripts" / "lez" / "pipeline.py").is_file()), None)
if WORK is None:
    raise FileNotFoundError("Không tìm thấy thư mục work của dự án.")
if str(WORK / "scripts") not in sys.path:
    sys.path.insert(0, str(WORK / "scripts"))
from lez.pipeline import preflight, run_pipeline, status, output_path


## 1. Chọn phạm vi và kiểm tra đầu vào

`TILE_IDS=None` xử lý toàn bộ 183 ô. Muốn xem một nhóm nhỏ, đặt danh sách mã ô; kết quả nhóm nhỏ không thay liên kết mạng toàn thành phố.


In [2]:
TILE_IDS = None
# Ví dụ: TILE_IDS = ["HN_R011_C012"]
CHECK = preflight(WORK, tile_ids=TILE_IDS)
print(json.dumps(CHECK, ensure_ascii=False, indent=2))


{
  "snapshot_utc": "2026-10-01T00:00:00Z",
  "active_tiles": 183,
  "selected_tiles": 183,
  "is_full_scope": true,
  "raw_hashes_and_manifests_match": true,
  "qgis": "4.2.3-Belém do Pará",
  "lez_polygon_configured": false,
  "network_calls": 0,
  "raw_osm_modified": false
}


## 2. Chạy hoặc tiếp tục

Module kiểm tra nguồn/cache, cập nhật tiến độ và lưu từng giai đoạn hoàn chỉnh. Bước mạng không gửi HTTP. Ranh giới Hà Nội hiện có đủ để dựng mạng; phân tầng LEZ/V/N được báo cáo riêng nếu còn thiếu.


In [3]:
RESULT = run_pipeline(WORK, stage="network", tile_ids=TILE_IDS, fetch_sources=False)
NETWORK_DIR = output_path(RESULT, "network_dir", WORK)


[boundaries] Đọc kết quả ranh giới đã kiểm tra {'stage_a_complete': False}
[index] Đọc chỉ mục OSM đã kiểm tra {'unique_objects': 1691428}
[network] Đọc mạng đã kiểm tra {'routable_arcs': 708233, 'segments': 567934}


## 3. Xem tổng kết

Chiều dài mạng đếm đoạn duy nhất; hai cung của đường hai chiều không được cộng thành hai lần chiều dài đường. `scope_parts` giữ phần đoạn nằm trong Hà Nội cùng liên kết về đoạn gốc.


In [4]:
summary_keys = ["selected_tiles", "is_full_scope", "highway_ways", "segments", "scope_parts", "arcs", "routable_arcs", "segment_statuses", "restriction_statuses", "graph", "verified_allowed_segments", "network_status"]
print(json.dumps({k: RESULT["network"][k] for k in summary_keys}, ensure_ascii=False, indent=2))
print("Đã dùng cache:", RESULT["cache_hits"])


{
  "selected_tiles": 183,
  "is_full_scope": true,
  "highway_ways": 254379,
  "segments": 567934,
  "scope_parts": 448317,
  "arcs": 867630,
  "routable_arcs": 708233,
  "segment_statuses": {
    "excluded": 75021,
    "provisionally_allowed": 465316,
    "review_required": 27597
  },
  "restriction_statuses": {
    "not_applicable": 695,
    "resolved": 253,
    "review_required": 212
  },
  "graph": {
    "nodes": 306525,
    "arcs": 708233,
    "turn_rules": 192,
    "weak_components": 2415,
    "largest_weak_component_nodes": 290827
  },
  "verified_allowed_segments": 0,
  "network_status": "provisional_osm_motorcycle_network"
}
Đã dùng cache: {'boundaries': True, 'index': True, 'network': True}


## 4. Xem các bảng

`segments` chứa đoạn duy nhất; `arcs` chứa chiều đi; `restrictions` và `turn_rules` giữ thông tin hạn chế rẽ. `controls` chứa barrier/tín hiệu và ghi nhận điểm có nằm trên way hay không.


In [5]:
segments_preview = pd.read_csv(NETWORK_DIR / "segments.csv", nrows=10, keep_default_na=False)
segments_preview


,segment_id,way_id,start_index,end_index,from_node,to_node,length_m,scope_length_m,access_status,rc,reason,tags_json,tile_ids
0,w9566542:0:1,9566542,0,1,osm/1893253381,osm/1893253440,46.258412,46.258412,excluded,other,research_excluded_highway:footway,"{""bridge"":""yes"",""highway"":""footway"",""layer"":""1...",HN_R011_C012
1,w9656653:0:1,9656653,0,1,osm/81794516,osm/84786846,115.073651,115.073651,review_required,RC4,restriction/16813020:conditional_turn_not_eval...,"{""hgv"":""no"",""highway"":""residential"",""maxspeed""...",HN_R011_C012
2,w9656730:0:3,9656730,0,3,osm/75621001,osm/6603790705,1183.184094,1183.184094,provisionally_allowed,RC1,osm_motorcycle=yes,"{""bicycle"":""no"",""bridge"":""yes"",""bus:conditiona...",HN_R011_C012
3,w9963509:0:1,9963509,0,1,osm/103015876,osm/6600775474,14.741113,14.741113,provisionally_allowed,RC3,road_class_default_not_field_verified,"{""highway"":""tertiary"",""lanes"":""2"",""lit"":""yes"",...",HN_R011_C012
4,w9963509:1:2,9963509,1,2,osm/6600775474,osm/11679590132,57.744415,57.744415,provisionally_allowed,RC3,road_class_default_not_field_verified,"{""highway"":""tertiary"",""lanes"":""2"",""lit"":""yes"",...",HN_R011_C012
5,w9963509:2:3,9963509,2,3,osm/11679590132,osm/11675900182,8.384919,8.384919,provisionally_allowed,RC3,road_class_default_not_field_verified,"{""highway"":""tertiary"",""lanes"":""2"",""lit"":""yes"",...",HN_R011_C012
6,w9963509:3:4,9963509,3,4,osm/11675900182,osm/2964574849,29.315573,29.315573,provisionally_allowed,RC3,road_class_default_not_field_verified,"{""highway"":""tertiary"",""lanes"":""2"",""lit"":""yes"",...",HN_R011_C012
7,w9963509:4:5,9963509,4,5,osm/2964574849,osm/1954877847,24.276383,24.276383,provisionally_allowed,RC3,road_class_default_not_field_verified,"{""highway"":""tertiary"",""lanes"":""2"",""lit"":""yes"",...",HN_R011_C012
8,w9964440:0:2,9964440,0,2,osm/6658453755,osm/12951168004,63.285521,63.285521,provisionally_allowed,RC3,road_class_default_not_field_verified,"{""highway"":""tertiary"",""lanes"":""2"",""lit"":""yes"",...",HN_R011_C012
9,w9964440:2:3,9964440,2,3,osm/12951168004,osm/12951168005,77.601678,77.601678,provisionally_allowed,RC3,road_class_default_not_field_verified,"{""highway"":""tertiary"",""lanes"":""2"",""lit"":""yes"",...",HN_R011_C012


In [6]:
restrictions = pd.read_csv(NETWORK_DIR / "restrictions.csv", keep_default_na=False)
restrictions.loc[restrictions["status"] == "review_required", ["restriction_id", "reason"]].head(30)


,restriction_id,reason
8,9772435,conditional_turn_not_evaluated
27,9824463,via_way_state_not_implemented
45,9854740,via_way_state_not_implemented
46,9854741,via_way_state_not_implemented
47,9854780,conditional_turn_not_evaluated
50,9857646,via_way_state_not_implemented
59,9861573,via_way_state_not_implemented
61,9870423,via_way_state_not_implemented
66,9870659,via_way_state_not_implemented
71,9871072,via_way_state_not_implemented


## 5. Kiểm tra graph có xét hạn chế rẽ

Dùng `MotorcycleNetwork` để đọc graph và kiểm tra chuỗi cung. Graph đã loại các cung còn cần xác minh; trạng thái cung đi vào phải được giữ khi xét bước đi tiếp.


In [7]:
from lez.graph import MotorcycleNetwork
GRAPH = MotorcycleNetwork.load(NETWORK_DIR / "motorcycle_graph.json.gz")
print("Số cung trong graph:", len(GRAPH.arcs))
# Với một chuỗi arc_id đã có: GRAPH.validate_walk(arc_ids)


Số cung trong graph: 708233


## 6. Báo cáo và chạy tiếp

Mở `work/data/processed/network.gpkg` trong QGIS. Báo cáo mới nhất nằm trong `work/reports/network`; lỗi hình học nằm trong `network_issues.csv` của phiên bản mạng. Chạy lại cell 2 để tiếp tục và đọc cache hợp lệ.


In [8]:
print("Bước A hoàn thành:", RESULT["stage_a_complete"])
print("Cần bổ sung:", len(RESULT["boundaries"]["needs_input"]))
print("Lỗi đầu mút cung:", RESULT["network"]["dangling_arc_endpoints"])
print("Lỗi quy tắc rẽ:", RESULT["network"]["invalid_turn_rules"])
print("Vấn đề hình học:", RESULT["network"]["zero_or_ambiguous_geometry_issues"])


Bước A hoàn thành: False
Cần bổ sung: 17
Lỗi đầu mút cung: 0
Lỗi quy tắc rẽ: 0
Vấn đề hình học: 1


## Tiếp tục theo LEZ 2030 và chỉ tiêu 20/40

Người nghiên cứu đã chọn kịch bản 2030. Mở `work/notebooks/05_sampling_2030.ipynb` để lấy chỉ báo dân cư/hoạt động, dựng phạm vi lấy mẫu linh hoạt quanh ranh giới, cắt mạng và sinh 20 tuyến ứng viên trong/40 ngoài. Bộ mới giữ lớp ranh giới đối chiếu và OSM gốc; phương án phạm vi lấy mẫu/tầng được ghi riêng.

Số tuyến được đếm từ hình học đã tạo, không phải số dòng cấu hình hoặc 20/40 ô V/N × RC. Quyền đi thực địa, điểm neo an toàn và chọn 30 tuyến đo còn là bước sau. Hướng dẫn: `work/docs/lez/SAMPLING_2030.md`.
